## 07 — Confidence thresholds (client-side guardrail)

**Problem:** a 2-class model *always* answers `chihuahua` or `muffin`.
A tea cup or an owl still gets one of those labels — often with medium confidence.

**Goal:** reject weak predictions before you trust them.

We apply two simple rules on the softmax probabilities:

1. **Confidence threshold** — `max(prob) >= CONF_MIN`
2. **Margin** — `top1 - top2 >= MARGIN_MIN` (how clear the winner is)

If either rule fails → label = `uncertain` (ask a human / try another photo).

This is **client-side** (your notebook / app). It does not change the model on the server.

**Endpoint (validated):** internal `:8888` (not the UI `:8080`).

In [ ]:
import json
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import requests
from PIL import Image

warnings.filterwarnings("ignore", message="Unverified HTTPS request")

MODEL = "muffin-chihuahua"
CLASSES = ["chihuahua", "muffin"]
EP = "http://muffin-chihuahua-predictor.chihuahua-vs-muffin-jan.svc.cluster.local:8888"
SESSION = requests.Session()

# --- Tunable guardrails ---
CONF_MIN = 0.80
MARGIN_MIN = 0.25

print("EP:", EP)
print("CONF_MIN:", CONF_MIN, "| MARGIN_MIN:", MARGIN_MIN)
t0 = time.perf_counter()
r = SESSION.get(f"{EP}/v2/models/{MODEL}", timeout=10)
print(f"metadata {r.status_code} in {time.perf_counter()-t0:.2f}s")
r.raise_for_status()



In [ ]:
def preprocess(path: Path) -> np.ndarray:
    img = Image.open(path).convert("RGB").resize((224, 224))
    arr = np.asarray(img).astype("float32") / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype="float32")
    std = np.array([0.229, 0.224, 0.225], dtype="float32")
    arr = (arr - mean) / std
    return np.transpose(arr, (2, 0, 1))[None, ...]


def softmax(logits: np.ndarray) -> np.ndarray:
    x = logits.astype("float32")
    e = np.exp(x - x.max())
    return e / e.sum()


def infer_raw(path: Path) -> np.ndarray:
    """Call OVMS. Prefer binary tensor (fast); fall back to JSON if needed.

    Why binary? JSON with 1×3×224×224 floats is ~1–2 MB text and very slow
    to serialize. Binary FP32 is ~600 KB and much faster.
    """
    chw = preprocess(path).astype(np.float32)
    raw = chw.tobytes(order="C")
    header = {
        "inputs": [
            {
                "name": "input",
                "shape": list(chw.shape),
                "datatype": "FP32",
                "parameters": {"binary_data_size": len(raw)},
            }
        ]
    }
    header_bytes = json.dumps(header).encode("utf-8")
    url = f"{EP}/v2/models/{MODEL}/infer"
    headers = {
        "Content-Type": "application/octet-stream",
        "Inference-Header-Content-Length": str(len(header_bytes)),
    }
    r = SESSION.post(url, data=header_bytes + raw, headers=headers, timeout=30)
    if r.status_code >= 400:
        # Fallback: classic JSON (slower)
        payload = {
            "inputs": [
                {
                    "name": "input",
                    "shape": list(chw.shape),
                    "datatype": "FP32",
                    "data": chw.ravel().tolist(),
                }
            ]
        }
        r = SESSION.post(
            url,
            headers={"Content-Type": "application/json"},
            data=json.dumps(payload),
            timeout=60,
        )
    r.raise_for_status()
    body = r.content
    # Response may be JSON or binary-framed; try JSON first
    try:
        out = np.array(r.json()["outputs"][0]["data"], dtype="float32")
    except Exception:
        # Minimal binary response parse: last 2 float32 values
        out = np.frombuffer(body[-8:], dtype="<f4")
    return softmax(out)


def decide(probs: np.ndarray) -> dict:
    order = np.argsort(probs)[::-1]
    i1, i2 = int(order[0]), int(order[1])
    conf = float(probs[i1])
    margin = float(probs[i1] - probs[i2])
    raw_label = CLASSES[i1]
    reasons = []
    if conf < CONF_MIN:
        reasons.append(f"confidence {conf:.1%} < {CONF_MIN:.0%}")
    if margin < MARGIN_MIN:
        reasons.append(f"margin {margin:.1%} < {MARGIN_MIN:.0%}")
    if reasons:
        return {
            "label": "uncertain",
            "raw_label": raw_label,
            "confidence": conf,
            "margin": margin,
            "probs": {CLASSES[i]: float(probs[i]) for i in range(len(CLASSES))},
            "accepted": False,
            "reason": "; ".join(reasons),
        }
    return {
        "label": raw_label,
        "raw_label": raw_label,
        "confidence": conf,
        "margin": margin,
        "probs": {CLASSES[i]: float(probs[i]) for i in range(len(CLASSES))},
        "accepted": True,
        "reason": "ok",
    }


def predict(path: Path) -> dict:
    t0 = time.perf_counter()
    result = decide(infer_raw(path))
    result["path"] = path
    result["latency_s"] = round(time.perf_counter() - t0, 2)
    return result


def show(result: dict):
    img = Image.open(result["path"]).convert("RGB")
    accepted = result["accepted"]
    color = "green" if accepted else "darkorange"
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
    axes[0].imshow(img)
    axes[0].axis("off")
    axes[0].set_title(
        f"{result['label']}  (raw={result['raw_label']})\n"
        f"conf={result['confidence']:.1%}  margin={result['margin']:.1%}  "
        f"({result['latency_s']}s)\n{result['reason']}",
        color=color,
        fontsize=10,
    )
    labels = list(result["probs"].keys())
    values = list(result["probs"].values())
    bar_colors = ["#4C9AFF" if lb == result["raw_label"] else "#D2D2D2" for lb in labels]
    axes[1].barh(labels, values, color=bar_colors)
    axes[1].axvline(CONF_MIN, color="red", linestyle="--", linewidth=1, label=f"CONF_MIN={CONF_MIN:.0%}")
    axes[1].set_xlim(0, 1)
    axes[1].set_xlabel("probability")
    axes[1].legend(loc="lower right", fontsize=8)
    for i, v in enumerate(values):
        axes[1].text(v + 0.02, i, f"{v:.1%}", va="center")
    fig.suptitle("ACCEPTED" if accepted else "UNCERTAIN — do not trust", color=color)
    fig.tight_layout()
    plt.show()
    return result



### A) In-domain smoke test (should usually be ACCEPTED)

Uses images from the lab dataset already on the workbench.

In [ ]:
# Fast smoke test: 1 muffin + 1 chihuahua (was 2+2 — JSON payload made that slow)
root = Path("/opt/app-root/src/data/muffin-chihuahua/test")
in_domain = (
    list(root.joinpath("muffin").glob("*.jpg"))[:1]
    + list(root.joinpath("chihuahua").glob("*.jpg"))[:1]
)

for p in in_domain:
    print("===", p.name, "===")
    t0 = time.perf_counter()
    result = predict(p)
    print(f"infer+guardrail {result['latency_s']}s | total cell step {time.perf_counter()-t0:.2f}s")
    show(result)



### B) Out-of-domain test (should often be UNCERTAIN)

1. Upload any non-muffin / non-chihuahua photo into the workbench, e.g.:
   `/opt/app-root/src/ood/` (tea cup, cat, car, screenshot…)
2. Set `OOD_DIR` below and run.

If everything is still ACCEPTED with high confidence, **raise** `CONF_MIN` / `MARGIN_MIN` and re-run.

In [ ]:
OOD_DIR = Path("/opt/app-root/src/ood")
OOD_DIR.mkdir(parents=True, exist_ok=True)

ood_files = [
    p
    for p in OOD_DIR.rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
]

if not ood_files:
    print(f"No images in {OOD_DIR}")
    print("Upload a few out-of-domain photos there, then re-run this cell.")
else:
    print(f"Found {len(ood_files)} OOD image(s)")
    for p in ood_files:
        print("===", p.name, "===" )
        show(predict(p))

### C) Quick tune helper

Change thresholds and re-score the same folders without rewriting functions.

In [ ]:
# Re-tune here, then re-run A / B cells
CONF_MIN = 0.85
MARGIN_MIN = 0.30
print("New thresholds:", CONF_MIN, MARGIN_MIN)

def summarize(paths):
    rows = []
    for p in paths:
        r = predict(p)
        rows.append(r)
        flag = "OK " if r["accepted"] else "UNC"
        print(
            f"{flag}  {p.name:30} raw={r['raw_label']:10} "
            f"conf={r['confidence']:.1%} margin={r['margin']:.1%} → {r['label']}"
        )
    return rows

print("\n-- in-domain --")
summarize(in_domain)
print("\n-- out-of-domain --")
summarize(ood_files) if ood_files else print("(none yet)")